# mHC-Pytorch 

mHC paper: [mHC: Manifold-Constrained Hyper-Connections](https://arxiv.org/abs/2512.24880)

blog: [【手撕 mHC】详解DeepSeek残差链接mHC进化之路（超长文、附代码）](https://zhuanlan.zhihu.com/p/1990683672337223894)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(42)

In [2]:
# config 

dim = 512
rate = 2
layer_id = 10
dynamic = True

bsz = 1
seq_len = 16

## Manifold-Constrained Hyper-Connections

### Math

\begin{equation}
    \begin{cases}
        \vec{\mathbf{x}}'_l = \text{RMSNorm}(\vec{\mathbf{x}}_l) \\
        \tilde{\mathcal{H}}_l^\text{pre} = \alpha_l^\mathrm{pre} \cdot (\vec{\mathbf{x}}'_l\phi^\mathrm{pre}_l) + \mathbf{b}_l^\mathrm{pre} \\
        \tilde{\mathcal{H}}_l^\text{post} = \alpha_l^\mathrm{post} \cdot (\vec{\mathbf{x}}'_l\phi^\mathrm{post}_l) + \mathbf{b}_l^\mathrm{post} \\
        \tilde{\mathcal{H}}_l^\text{res} = \alpha_l^\mathrm{res} \cdot \text{mat}(\vec{\mathbf{x}}'_l\phi^\mathrm{res}_l) + \mathbf{b}_l^\mathrm{res}, \\
    \end{cases}
\end{equation}



\begin{equation}
    \begin{cases}
        \mathcal{H}_l^\text{pre} = \sigma(\tilde{\mathcal{H}}_l^\text{pre}) \\
        \mathcal{H}_l^\text{post} = 2\sigma(\tilde{\mathcal{H}}_l^\text{post}) \\
        \mathcal{H}_l^\text{res} = \text{Sinkhorn-Knopp}(\tilde{\mathcal{H}}_l^\text{res}),
    \end{cases}
\end{equation}


### sinkhorn_knopp 归一化

原论文

To this end, we restrict $\mathcal{H}^\text{res}_{l}$ to be a doubly stochastic matrix, which has non-negative entries where both the rows and columns sum to 1. Formally, let $\mathcal{M}^\mathrm{res}$ denote the manifold of doubly stochastic matrices (also known as the Birkhoff polytope).
We constrain $\mathcal{H}^\text{res}_{l}$ to $\mathcal{P}_{\mathcal{M}^\mathrm{res}}(\mathcal{H}^\text{res}_{l})$, defined as:
$$
\begin{equation}
    \mathcal{P}_{\mathcal{M}^\mathrm{res}}(\mathcal{H}^\text{res}_{l}) \coloneq \left\{ \mathcal{H}^\text{res}_{l} \in \mathbb{R}^{n \times n} \mid \mathcal{H}^\text{res}_{l}\mathbf{1}_n = \mathbf{1}_n, \ \mathbf{1}^\top_n\mathcal{H}^\text{res}_{l} = \mathbf{1}^\top_n, \ \mathcal{H}^\text{res}_{l} \geq 0 \right\},
\end{equation}
$$
where $\mathbf{1}_n$ represents the $n$-dimensional vector of all ones.

In [ ]:
def sinkhorn_knopp_batched(A, it=1000, eps=1e-8):
    f"""
    A is not negative matrix
    """

    batch_size, n, _, = A.shape # [B*L, N, N]

    u = torch.ones(batch_size, n)  # 行缩放向量: [B*L, N]
    v = torch.ones(batch_size, n)  # 列缩放向量: [B*L, N]

    # 交替归一化行和列
    for _ in range(it):
        # 列归一化: v = 1 / (A^T @ u)
        v_temp = v.unsqueeze(2)  # [B*L, N, 1]
        Av = torch.bmm(A, v_temp).squeeze(2)  # [B*L, N]
        u = 1.0 / (Av + eps)  # [B*L, N]
        
        u_temp = u.unsqueeze(2)  # (B, n, 1)
        At_u = torch.bmm(A.transpose(1, 2), u_temp).squeeze(2)
        v = 1.0 / (At_u + eps)

    U = torch.diag_embed(u)  # (B, n, n)
    V = torch.diag_embed(v)  # (B, n, n)
    P = torch.bmm(torch.bmm(U, A), V)

    return P, U, V

    _, U, V = res_norm(H_res_exp.reshape(B*L, N, N), self.max_sk_it)
    P = torch.bmm(torch.bmm(U.detach(), H_res_exp.reshape(B*L, N, N)), V.detach())

In [4]:
A = torch.randn(2,3,3)
A = A.exp() # NOT negative trick
# example1
P, _, _, = sinkhorn_knopp_batched(A, it=2)
print(P.shape)
print('it=2\t', P[0].sum(dim=0), P[0].sum(dim=1))

# example2
P, _, _, = sinkhorn_knopp_batched(A, it=20)
print('it=20\t', P[0].sum(dim=0), P[0].sum(dim=1))

torch.Size([2, 3, 3])
it=2	 tensor([1.0000, 1.0000, 1.0000]) tensor([0.9788, 1.0334, 0.9878])
it=20	 tensor([1.0000, 1.0000, 1.0000]) tensor([1., 1., 1.])


### Fuse kernel

$$
\begin{align}
    \phi_l                                                                      &: \text{tfloat32}          &&[nC, n^2+2n]                                                             \\
    \vec{\mathbf{x}}_l                                                                      &: \text{bfloat16}          &&[1, nC]                                                                     \\
    \alpha_l^\mathrm{pre}, \alpha_l^\mathrm{post}, \alpha_l^\mathrm{res}                                                    &: \text{float32}           &&\text{Scalars}                                                         \\
    \mathbf{b}_l                                                                      &: \text{float32}           &&[1, n^2+2n]                                                                  \\
    \left[{\tilde{\tilde{\mathcal{H}}}^{\mathrm{pre}}_{l}}, {\tilde{\tilde{\mathcal{H}}}^{\mathrm{post}}_{l}}, {\tilde{\tilde{\mathcal{H}}}^{\mathrm{res}}_{l}}\right]   &: \text{float32}           &&= \vec{\mathbf{x}}_l\phi_l                                                  \\
    r                                                                               &: \text{float32}           &&= \left\|\vec{\mathbf{x}}_l\right\|_2 / \sqrt{nC}  ;  \text{——RMS:}r = \frac{1}{RMS} = \frac{||\vec{\mathbf{x}}_l||}{\sqrt{nC}}                                              \\
    \left[\tilde{\mathcal{H}}^{\mathrm{pre}}_{l}, \tilde{\mathcal{H}}^{\mathrm{post}}_{l}, \tilde{\mathcal{H}}^{\mathrm{res}}_{l}\right]         &: \text{float32}           &&= 1/r \left[\alpha_l^\mathrm{pre}{\tilde{\tilde{\mathcal{H}}}^{\mathrm{pre}}_{l}}, \alpha_l^\mathrm{post}{\tilde{\tilde{\mathcal{H}}}^{\mathrm{post}}_{l}}, \alpha_l^\mathrm{res}{\tilde{\tilde{\mathcal{H}}}^{\mathrm{res}}_{l}}\right] + \mathbf{b}_l \\
    \mathcal{H}^{\mathrm{pre}}_{l}                                                                      &: \text{float32}           &&= \sigma\left(\tilde{\mathcal{H}}^{\mathrm{pre}}_{l}\right)                                   \\
    \mathcal{H}^{\mathrm{post}}_{l}                                                                      &: \text{float32}           &&= 2\sigma\left(\tilde{\mathcal{H}}^{\mathrm{post}}_{l}\right)                                  \\
    \mathcal{H}^{\mathrm{res}}_{l}                                                                      &: \text{float32}           &&= \text{Sinkhorn-Knopp}\left(\tilde{\mathcal{H}}^{\mathrm{res}}_{l}\right)   
\end{align}
$$



原本的 RmsNorm 计算流程为:

$$
\hat{x} =\gamma \frac{x}{\text{RMS}} \\
H = \phi\hat{x}
$$

对应的 torch 实现

In [5]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-12):
        super(RMSNorm, self).__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        mean = (x**2).mean(-1, keepdim=True)
        out_mean = x / torch.sqrt(mean + self.eps) # root mean square
        out = self.gamma * out_mean 
        return out


观察到 mHC 中的 RMSNorm 在处理高维隐藏状态时会带来显著延迟，我们将除以范数的操作调整到矩阵乘法之后。这一优化在保持数学等价性的同时提高了效率。

fuse RMSNorm after matrix multiple($\vec{\mathbf{x}}_l\phi_l$)

$$
\begin{align} \tilde{x} &= \phi\gamma x, x\in\mathbb{R}^{nC}  \\ H &=  \tilde{x} \frac{1}{\text{RMS}(x)}=\tilde{x}\frac{1}{r} \\ \frac{1}{\text{RMS}} &= \frac{1}{\sqrt{ \frac{1}{nC}(\sum_{j=1}^{nC} x_j^2)}} = \frac{1}{\sqrt{ \frac{1}{nC}}\sqrt{(\sum_{j=1}^{nC} x_j^2)}} \\ &=\frac{1}{\sqrt{\frac{1}{nC}}} \frac{1}{\sqrt{(\sum_{j=1}^{nC} x_j^2)}} \\ &=\sqrt{nC}\frac{1}{\vert\vert x\vert\vert_2} \\ r = \text{RMS} &= \frac{\vert\vert x\vert\vert_2}{\sqrt{nC}} \end{align}
$$

```
原始公式:
    H = RMSNorm(x) @ W
      = (γ · x / RMS(x)) @ W

关键观察: γ 是常数，1/RMS 是标量
    H = (γ · x · 1/RMS) @ W
      = (x @ W) · (γ/RMS)      ← 提取到 GEMM 之后!

如果 γ 已融入权重 W (W' = γ·W): (本文代码未实现)
    H = (x @ W') / RMS
      = (x @ W') · rsqrt(Σx²/n + ε)

所以可以:
    1. 并行算: GEMM: H̃ = x @ W  和  sqrsum = Σxᵢ²
    2. 最后乘: H = H̃ · rsqrt(sqrsum/n + ε)
```


### ManifoldHyperConnectionFuses实现

In [ ]:
import math
class ManifoldHyperConnectionFuse(nn.Module):
    """
    h: hyper hidden matrix (BxLxNxD)
        B: batch_size
        L: Seq_len
        N: expansion rate
        D: feature dim
    """
    def __init__(self, dim, rate, layer_id, max_sk_it):
        super(ManifoldHyperConnectionFuse, self).__init__()

        self.n = rate  # 分支数，如 rate=2
        self.dim = dim  # 特征维度，如 dim=512

        self.nc = self.n * self.dim  # 展平后维度 = 2*512 = 1024
        self.n2 = self.n * self.n  # n² = 4 (用于 H_res 矩阵)

        # norm flatten, 代码中仅使用 RMSNorm 的 gamma 参数
        self.norm = RMSNorm(dim*rate)

        # parameters(可学习参数)
        self.w = nn.Parameter(torch.zeros(self.nc, self.n2 + 2*self.n))  # [nc, n²+2n] = [1024, 8]
        self.alpha = nn.Parameter(torch.ones(3) * 0.01)  # 缩放因子: [3]
        self.beta = nn.Parameter(torch.zeros(self.n2 + 2*self.n) * 0.01)  # 偏置: [n²+2n]

        # max sinkhorn knopp iterations
        self.max_sk_it = max_sk_it

    # mapping 函数：计算三个超连接矩阵
    #            h [B, L, N, D]
    #                  │
    #                  ↓ 展平
    #            h_vec [B, L, N*D]
    #                  │
    #                  ↓ @ w
    #            H [B, L, n²+2n] 
    #                  │ * alpha + beta
    #     ┌────────────┬────────────┐
    #     ↓            ↓            ↓
    # H_pre [n]    H_post [n]   H_res [n²]
    #     ↓            ↓            ↓
    # sigmoid      2*sigmoid   Sinkhorn-Knopp
    #     ↓            ↓            ↓
    #   [0,1]        [0,2]      双随机矩阵(行列和=1)
    #
    # 三个超连接矩阵的含义:
    #  矩阵	     形状	           约束	                     作用
    # H_pre	  [B, L, N]	    sigmoid → [0,1]	    控制各分支进入功能层前的权重
    # H_post  [B, L, N]     2×sigmoid → [0,2]	控制功能层输出分配给各分支的权重
    # H_res	  [B, L, N, N]	双随机矩阵	          控制分支间残差信息的交换
    #
    # 双随机矩阵的约束：每行和 = 1, 每列和 = 1, 所有元素 ≥ 0。这确保了残差混合是"公平"的，即信息既不丢失也不放大，保持特征流守恒! 
    def mapping(self, h, res_norm):
        B, L, N, D = h.shape  # [batch, seq_len, rate, dim]

        # 1.vectorize: 展平多分支特征
        # 向量化目的：对多个分支的输入拼接 flat 化后学习，某种程度上，扩展了特征向量维度，大大增加了存储容量。
        h_vec_flat = h.reshape(B, L, N*D)

        # RMSNorm Fused Trick: gamma-scaling
        # h_vec_flat 被读取 2 次 (乘 gamma 一次，算 norm 一次)。
        # 不能移动到矩阵乘之后，因为 γ · x 是在 [N*D] 维度上逐元素乘, 即 γ:[N*D], x: [B, L, N*D]， 而H:[B, L, n²+2n]
        # 但可以融合到 w 中，减少一次读取
        # W_fused = torch.diag(gamma) @ W   # 或 gamma.unsqueeze(-1) * W
        # H = x @ W_fused   # 直接用 W_fused, 不需要单独乘 gamma
        h_vec = self.norm.gamma * h_vec_flat

        # 2.projection
        H = h_vec @ self.w  # [B, L, N*D] @ [N*D, n²+2n] = [B, L, n²+2n]

        # RMSNorm Fused: compute r
        r = h_vec_flat.norm(dim=-1, keepdim=True) / math.sqrt(self.nc)
        r_ = 1.0 / r  # [B, L, n²+2n]

        # 4. mapping: 分离并处理三个超连接矩阵
        n = N
        H_pre = r_ * H[:,:, :n] * self.alpha[0] + self.beta[:n]  # [B, L, N]
        H_post = r_ * H[:,:, n:2*n] * self.alpha[1] + self.beta[n:2*n]  # [B, L, N]
        H_res = r_ * H[:,:, 2*n:] * self.alpha[2] + self.beta[2*n:]  # [B, L, N*N]

        # 5. final constrained mapping: 约束映射
        H_pre = F.sigmoid(H_pre)  # [0, 1] 范围
        H_post = 2 * F.sigmoid(H_post)  # [0, 2] 范围，2 * sigmoid 使输出范围得到一定的扩展

        # 6. sinkhorn_knopp iteration: H_res 通过 Sinkhorn-Knopp 变成双随机矩阵
        H_res = H_res.reshape(B, L, N, N)
        H_res_exp = H_res.exp()  # 先 exp 保证非负
        with torch.no_grad():
            # res_norm 为 sinkhorn_knopp_batched 方法, 返回双随机矩阵
            _, U, V = res_norm(H_res_exp.reshape(B*L, N, N), self.max_sk_it)
        # recover
        P = torch.bmm(torch.bmm(U.detach(), H_res_exp.reshape(B*L, N, N)), V.detach())
        H_res = P.reshape(B, L, N, N)

        return H_pre, H_post, H_res

    # process 函数：对 h: [B, L, N, D]执行超连接
    #
    # h [N个分支]        H_pre [1, N]              h_pre [1个输出]
    # ┌───┐             ┌───────┐                 ┌───┐
    # │ h1│             │w1  w2 │                 │   │
    # │ h2│     @       └───────┘          =      │ Σ │
    # └───┘                                       └───┘
    #                     加权合并: h_pre = w1*h1 + w2*h2
    #
    # h [N个分支]        H_res [N, N]               h_res [N个分支]
    # ┌───┐           ┌─────────────┐              ┌───┐
    # │ h1│           │ r11    r12  │              │h1'│
    # │ h2│    @      │ r21    r22  │      =       │h2'│
    # └───┘           └─────────────┘              └───┘
    #                 残差混合: h1' = r11*h1 + r12*h2
    #                          h2' = r21*h1 + r22*h2
    #
    # h_pre: [B, L, N] -> [B, L, 1, D]
    # h_res: [B, L, N, N] -> [B, L, N, D]
    def process(self, h, H_pre, H_res):
        # 1. Pre-connection: 多分支加权合并为单一输入
        # [B, L, N]-> [B, L, 1, N] @ [B, L, N, D] = [B, L, 1, D] → 合并后的单一特征
        h_pre = H_pre.unsqueeze(dim=2) @ h
        # ==== 等价的 elementwise 写法 ====
        # (h * H_pre.unsqueeze(-1)): [B, L, N, D] * [B, L, N, 1] -> [B, L, N, D]
        #   .sum(dim=-2): 在 N 维度求和 -> [B, L, 1, D]
        # h_pre = (h * H_pre.unsqueeze(dim=-1)).sum(dim=-2, keepdim=True)

        # 2. Residual-connection: 分支间残差混合
        # [B, L, N, N] @ [B, L, N, D] = [B, L, N, D] → 混合后的各分支残差
        h_res = H_res @ h
        # ==== 等价的 elementwise 写法 ====
        # 对于输出的每个位置 (b, l, i, d):
        #   h_res[b, l, i, d] = Σ_j H_res[b, l, i, j] * h[b, l, j, d]
        # 使用 einsum 实现 (比纯 elementwise 更清晰):
        # h_res = torch.einsum('blij,bljd->blid', H_res, h)
        # 或者用广播+求和 (纯 elementwise):
        # h_res = (H_res.unsqueeze(-1) * h.unsqueeze(-3)).sum(dim=-2)
        # 其中: H_res.unsqueeze(-1): [B,L,N,N,1], h.unsqueeze(-3): [B,L,1,N,D]
        #       广播后: [B,L,N,N,D], 在 j(dim=-2) 维度求和 -> [B,L,N,D]
        return h_pre, h_res

    def depth_connection(self, h_res, h_out, beta):
        # h_res: [B, L, N, D] - 残差混合后的各分支
        # h_out: [B, L, 1, D] - 功能层(Attn/FFN)的输出
        # beta: [B, L, N] - H_post 分配权重

        # 将 Attn/MLP 的输出按权重分配给各分支
        # ============ 矩阵乘写法 ============
        # [B, L, N, 1] @ [B, L, 1, D] = [B, L, N, D]
        post_mapping = beta.unsqueeze(dim=-1) @ h_out

        # ============ 等价的 elementwise 写法 ============
        # beta: [B, L, N] -> [B, L, N, 1]
        # h_out: [B, L, 1, D]
        # 广播乘法: [B, L, N, 1] * [B, L, 1, D] = [B, L, N, D]
        # post_mapping = beta.unsqueeze(dim=-1) * h_out

        # 与残差相加
        out = post_mapping + h_res
        return out


### fwd test

In [8]:
max_sk_it = 20
mHC = ManifoldHyperConnectionFuse(dim = dim, 
                                  rate = rate, 
                                  layer_id = layer_id,
                                  max_sk_it = max_sk_it)

attn = nn.Linear(dim, dim)

In [9]:
h = torch.randn(bsz, seq_len, rate, dim)
H_pre, H_post, H_res = mHC.mapping(h, sinkhorn_knopp_batched)
h_pre, h_res = mHC.process(h, H_pre, H_res)
h_out = attn(h_pre) 
out = mHC.depth_connection(h_res, h_out, beta=H_post)
print('out', out.shape)

out torch.Size([1, 16, 2, 512])


## Decoder Block

In [10]:
class DecoderBlockmHC(nn.Module):
    def __init__(self, dim, rate, layer_id, max_sk_it):
        super(DecoderBlockmHC, self).__init__()
        self.attn = nn.Linear(dim, dim)  # 用简单线性层模拟注意力计算
        self.attn_mHC = ManifoldHyperConnectionFuse(dim = dim, rate = rate, layer_id = layer_id, max_sk_it = max_sk_it)
        self.ffn = nn.Linear(dim, dim)  # 用简单线性层模拟mlp
        self.ffn_mHC = ManifoldHyperConnectionFuse(dim = dim, rate = rate, layer_id = layer_id, max_sk_it = max_sk_it)

    def forward(self, h):
        # h:[bsz, seq_len, rate, dim]
        # ========== Attention 部分 ==========
        # 输入 h [B, L, N, D]
        #          │
        #     ┌────┴────┐
        #     │ mapping │ → 得到 H_pre, H_post, H_res
        #     └────┬────┘
        #          │ h, H_pre, H_res
        #     ┌────┴────┐
        #     │ process │
        #     └────┬────┘
        #     ┌────┴────────────┐
        #     ↓                 ↓
        # h_pre [1,D]      h_res [N,D]
        #     ↓                 │
        # ┌────────┐            │
        # │  Attn  │            │
        # └───┬────┘            │
        #     ↓                 │
        # h_out [1,D]           │
        #     └────────┬────────┘
        #              │ H_post
        #     ┌────────┴────────┐
        #     │ depth_connection│
        #     └────────┬────────┘
        #              ↓
        #     输出 h [B, L, N, D]
        #
        # 1. 计算超连接矩阵
        H_pre, H_post, H_res = self.attn_mHC.mapping(h, sinkhorn_knopp_batched)
        # 2. 执行超连接
        # h_pre: [B, L, 1, D] - 送入 Attention 的输入, h_res: [B, L, N, D] - 各分支的残差
        h_pre, h_res = self.attn_mHC.process(h, H_pre, H_res)
        # 3. 执行 Attention 计算
        h_out = self.attn(h_pre)
        # 4. 深度连接, 输出: [B, L, N, D]
        h = self.attn_mHC.depth_connection(h_res, h_out, beta=H_post)

        # ========== FFN 部分 ==========
        # 同样的流程...
        H_pre, H_post, H_res = self.ffn_mHC.mapping(h, sinkhorn_knopp_batched)
        h_pre, h_res = self.ffn_mHC.process(h, H_pre, H_res)
        h_out = self.ffn(h_pre) 
        h = self.ffn_mHC.depth_connection(h_res, h_out, beta=H_post)
        return h

## Model

In [11]:
# LanguageModelmHC
#     │
#     ├── Embedding: [bsz, seq_len] → [bsz, seq_len, dim]
#     │
#     ├── Repeat: [bsz, seq_len, dim] → [bsz, seq_len, rate, dim]  ← 关键！扩展为多分支
#     │
#     ├── DecoderBlockmHC × num_layers
#     │       │
#     │       ├── Attention + mHC
#     │       └── FFN + mHC
#     │
#     ├── Sum: [bsz, seq_len, rate, dim] → [bsz, seq_len, dim]  ← 多分支合并
#     │
#     └── LM_head: [bsz, seq_len, dim] → [bsz, seq_len, vocab_size]
class LanguageModelmHC(nn.Module):
    def __init__(self, num_layer, vocab_size, dim, rate, max_sk_it):
        super(LanguageModelmHC, self).__init__()
        self.n = rate
        self.embd = nn.Embedding(vocab_size, dim)
        self.decoder = nn.ModuleList(
            [ DecoderBlockmHC(dim, rate, layer_id, max_sk_it) for layer_id in range(num_layer) ]
        )
        self.lm_head = nn.Linear(dim, vocab_size)

    def forward(self, x):
        h = self.embd(x)  # [bsz, seq_len] → [bsz, seq_len, dim]

        # repeat h
        h=h.unsqueeze(dim=2)
        h = h.repeat(1,1,self.n,1)  # [bsz, seq_len, dim] → [bsz, seq_len, rate, dim] ← 关键！扩展为多分支

        # decoder forward
        for block in self.decoder:
            h = block(h)

        # sum transform branch
        h = h.sum(dim=2)  # [bsz, seq_len, rate, dim] → [bsz, seq_len, dim]  ← 多分支合并

        logits = self.lm_head(h)
        
        return logits

## mHC Model Forward

In [12]:
# model

vocab_size = 100
model =LanguageModelmHC(2, vocab_size, dim, rate, max_sk_it)
x = torch.randint(vocab_size, (bsz, seq_len))

In [13]:
logits = model(x)
print(logits.shape)

torch.Size([1, 16, 100])
